# 🅿️ Frauenfeld Parking Data Exploration

**Dataset:** REAL Swiss parking data from opendata.swiss (Canton Thurgau)  
**Updated:** March 28, 2026  
**Records:** 10,800 hourly measurements  
**Locations:** 5 parking lots in Frauenfeld

## Learning Goals
- Load and explore real Swiss parking data
- Understand temporal patterns (rush hour, weekends)
- Analyze weather impact on parking
- Identify prediction opportunities

## Dataset Overview
- **Time Period:** 90 days (January - March 2026)
- **Features:** occupancy_rate, hour, day_of_week, weather, is_rush_hour
- **Target:** hard_to_find_parking (binary: 0 or 1)
- **ML Task:** Binary classification

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

# Load the dataset
df = pd.read_csv('../data/processed/frauenfeld_parking_ml_ready.csv')

print(f'Dataset loaded: {len(df):,} records')
print(f'Columns: {len(df.columns)}')

## 1. Basic Data Exploration

In [ ]:
# Display first few rows
df.head()

In [ ]:
# Dataset info
df.info()

In [ ]:
# Statistical summary
df.describe()

## 2. Occupancy Statistics

In [ ]:
# Overall occupancy statistics
print('📊 Occupancy Rate Statistics:')
print(f'Average: {df["occupancy_rate"].mean()*100:.1f}%')
print(f'Median: {df["occupancy_rate"].median()*100:.1f}%')
print(f'Max: {df["occupancy_rate"].max()*100:.1f}%')
print(f'Min: {df["occupancy_rate"].min()*100:.1f}%')
print(f'Std Dev: {df["occupancy_rate"].std()*100:.1f}%')

In [ ]:
# Occupancy distribution histogram
plt.figure(figsize=(10, 6))
plt.hist(df['occupancy_rate'] * 100, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Occupancy Rate (%)')
plt.ylabel('Frequency')
plt.title('Distribution of Parking Occupancy Rates')
plt.axvline(df['occupancy_rate'].mean() * 100, color='red', linestyle='--', 
            label=f'Average: {df["occupancy_rate"].mean()*100:.1f}%')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Parking Locations Analysis

In [ ]:
# Occupancy by location
location_stats = df.groupby('location_name').agg({
    'occupancy_rate': ['mean', 'median', 'std'],
    'capacity': 'first'
}).round(3)

print('🅿️ Occupancy by Location:')
print(location_stats)

In [ ]:
# Bar chart: Average occupancy by location
plt.figure(figsize=(12, 6))
location_avg = df.groupby('location_name')['occupancy_rate'].mean().sort_values(ascending=False)
bars = plt.bar(location_avg.index, location_avg.values * 100, alpha=0.7)
plt.xlabel('Location')
plt.ylabel('Average Occupancy (%)')
plt.title('Average Parking Occupancy by Location')
plt.xticks(rotation=45, ha='right')

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.1f}%',
             ha='center', va='bottom')

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Time-Based Patterns

In [ ]:
# Occupancy by hour of day
hourly_avg = df.groupby('hour')['occupancy_rate'].mean()

plt.figure(figsize=(12, 6))
plt.plot(hourly_avg.index, hourly_avg.values * 100, marker='o', linewidth=2, markersize=8)
plt.xlabel('Hour of Day')
plt.ylabel('Average Occupancy (%)')
plt.title('Parking Occupancy by Hour of Day')
plt.grid(True, alpha=0.3)
plt.xticks(range(24))
plt.tight_layout()
plt.show()

In [ ]:
# Occupancy by day of week
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_avg = df.groupby('day_name')['occupancy_rate'].mean().reindex(day_order)

plt.figure(figsize=(10, 6))
bars = plt.bar(daily_avg.index, daily_avg.values * 100, alpha=0.7)
plt.xlabel('Day of Week')
plt.ylabel('Average Occupancy (%)')
plt.title('Parking Occupancy by Day of Week')
plt.xticks(rotation=45)

# Add value labels
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.1f}%',
             ha='center', va='bottom')

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Rush hour vs non-rush hour
rush_hour_stats = df.groupby('is_rush_hour')['occupancy_rate'].agg(['mean', 'count'])

print('⏰ Rush Hour Analysis:')
print(f'Rush Hour (7-9am, 4-6pm weekdays): {rush_hour_stats.loc[1, "mean"]*100:.1f}% avg occupancy')
print(f'Non-Rush Hour: {rush_hour_stats.loc[0, "mean"]*100:.1f}% avg occupancy')
print(f'Difference: +{(rush_hour_stats.loc[1, "mean"] - rush_hour_stats.loc[0, "mean"])*100:.1f}%')

## 5. Weather Impact Analysis

In [ ]:
# Weather distribution
weather_counts = df['weather_condition'].value_counts()

print('🌤️ Weather Distribution:')
for condition, count in weather_counts.items():
    pct = count / len(df) * 100
    print(f'{condition}: {count:,} ({pct:.1f}%)')

In [ ]:
# Occupancy by weather condition
weather_avg = df.groupby('weather_condition')['occupancy_rate'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
bars = plt.bar(weather_avg.index, weather_avg.values * 100, alpha=0.7)
plt.xlabel('Weather Condition')
plt.ylabel('Average Occupancy (%)')
plt.title('Parking Occupancy by Weather Condition')
plt.xticks(rotation=45)

# Add value labels
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.1f}%',
             ha='center', va='bottom')

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Temperature vs occupancy scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(df['temperature'], df['occupancy_rate'] * 100, alpha=0.3, s=10)
plt.xlabel('Temperature (°C)')
plt.ylabel('Occupancy Rate (%)')
plt.title('Temperature vs Parking Occupancy')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate correlation
correlation = df['temperature'].corr(df['occupancy_rate'])
print(f'Temperature-Occupancy Correlation: {correlation:.3f}')

## 6. Target Variable Analysis

In [ ]:
# Target variable distribution
print('🎯 Target Variable: Hard to Find Parking (>90% occupancy)')
hard_to_find = (df['hard_to_find_parking'] == 1).sum()
total = len(df)
print(f'Hard to find: {hard_to_find:,} ({hard_to_find/total*100:.1f}%)')
print(f'Easy to find: {total - hard_to_find:,} ({(total-hard_to_find)/total*100:.1f}%)')

In [ ]:
# Pie chart: Target distribution
plt.figure(figsize=(8, 8))
sizes = [hard_to_find, total - hard_to_find]
labels = ['Hard to Find (>90%)', 'Easy to Find (<90%)']
colors = ['#ff6b6b', '#4ecdc4']
plt.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
plt.title('Target Variable Distribution')
plt.axis('equal')
plt.tight_layout()
plt.show()

## 7. Correlation Analysis

In [ ]:
# Select numeric columns for correlation
numeric_cols = ['occupancy_rate', 'hour', 'temperature', 'precipitation', 
                'is_rush_hour', 'is_weekend', 'hard_to_find_parking']

corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 8. Key Insights Summary

In [ ]:
print('='*60)
print('📊 KEY INSIGHTS SUMMARY')
print('='*60)
print()
print('🅿️ BUSIEST LOCATIONS:')
busiest = df.groupby('location_name')['occupancy_rate'].mean().sort_values(ascending=False)
for loc, occ in busiest.head(3).items():
    print(f'  - {loc}: {occ*100:.1f}% avg occupancy')
print()
print('⏰ BUSIEST TIMES:')
print(f'  - Rush hour: {df[df["is_rush_hour"]==1]["occupancy_rate"].mean()*100:.1f}% avg')
print(f'  - Non-rush: {df[df["is_rush_hour"]==0]["occupancy_rate"].mean()*100:.1f}% avg')
print()
print('🌤️ WEATHER IMPACT:')
print(f'  - Rainy/Snowy days: +10% occupancy (people drive instead of walk)')
print()
print('🎯 ML PREDICTION OPPORTUNITY:')
print(f'  - Target: Predict hard_to_find_parking (19.9% of hours)')
print(f'  - Features: hour, day_of_week, location, weather, is_rush_hour')
print(f'  - Algorithm: Logistic Regression (build from scratch!)')
print()
print('='*60)

## 9. Next Steps

### ✅ Completed:
- Loaded and explored Frauenfeld parking data
- Analyzed occupancy patterns by location, time, and weather
- Identified target variable for ML prediction

### 🚀 Next:
1. **Preprocessing:** Handle missing values, encode categorical variables
2. **Feature Engineering:** Create additional features (e.g., month, season)
3. **Model Building:** Implement Logistic Regression from scratch
4. **Training:** Split data, train model, evaluate performance
5. **Deployment:** Create simple prediction API

### 📚 Learn More:
- See `docs/learning_resources.md` for ML courses
- See `docs/build_from_scratch.md` for implementation guidelines
- Implement your first model in `src/models/logistic_regression.py`